# 02 - Data Understanding
This notebook verifies provenance, physical structure, schemas, keys, quality, imbalance, time coverage and leakage risks before any feature engineering or modelling. Raw files remain unchanged.

In [1]:
from pathlib import Path
import csv, json, sys
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from project_paths import CRSS_YEARS, TABLE_DIR, complaint_file, crss_extracted_dir
pd.set_option('display.max_columns', 120)
PROJECT_ROOT

WindowsPath('C:/Users/delan/Downloads/NHTSA_Road_Vehicle_Safety_Capstone_Last_Lap')

## 1. Verified source inventory
The full standard-library audit reads all 140 CRSS CSV files and the complete complaint file. It records row counts, column counts, file sizes, headers and SHA-256 hashes. Re-run it only when the raw snapshot changes.

In [2]:
inventory = pd.read_csv(TABLE_DIR / 'crss_raw_file_inventory.csv')
complaint_audit = json.loads((TABLE_DIR / 'complaint_raw_file_audit.json').read_text(encoding='utf-8'))
display(inventory.groupby('year').agg(files=('table','size'), rows_across_tables=('rows','sum'), bytes=('size_bytes','sum')))
display(pd.Series(complaint_audit, name='value').to_frame())

,files,rows_across_tables,bytes
year,,,
2020,28,1987672,741079176
2021,28,1989565,745734450
2022,28,1969763,737972567
2023,28,1814963,684104538
2024,28,1876428,709161049


,value
file,C:\Users\delan\Downloads\NHTSA_Road_Vehicle_Sa...
rows,2250304
documented_columns,51
observed_row_widths,{'51': 2250304}
blank_narratives,39
blank_components,149
unique_odino,1622118
repeated_component_rows,628186
size_bytes,1605264894
sha256,ca2d3fe0c53a7b2ecc5a5354162914ae21e046d7766452...


### Audit finding requiring documentation
The complaint snapshot contains **2,250,304 rows**, matching the proposal. Every row has the documented 51-field width. An earlier audit reported 2,249,110 rows and two malformed widths; that was a parsing artefact. The NHTSA flat file is unquoted, and narratives contain literal `"` characters, so default CSV quote handling merged neighbouring records. The audit now reads the file with `quoting=csv.QUOTE_NONE`. The SHA-256 hash is unchanged, confirming the source file itself was never altered.

## 2. CRSS relational structure
- `accident`: one row per sampled crash; join key `CASENUM`.
- `vehicle`: one row per involved vehicle; composite key `CASENUM`, `VEH_NO`.
- `person`: one row per involved person; composite key `CASENUM`, `VEH_NO`, `PER_NO`.
- Remaining event/risk-factor tables have their own documented grains and must be aggregated before joining to crash level.

A direct accident-to-vehicle-to-person merge would multiply crashes and is prohibited.

In [3]:
core_summary = inventory[inventory['table'].isin(['accident','vehicle','person'])]
display(core_summary.pivot(index='year', columns='table', values='rows'))
schema_counts = core_summary.pivot(index='year', columns='table', values='columns')
display(schema_counts)

table,accident,person,vehicle
year,,,
2020,54745,131962,94718
2021,54200,133734,95785
2022,53955,132175,94756
2023,50103,122388,87461
2024,51658,126159,90641


table,accident,person,vehicle
year,,,
2020,80,108,167
2021,80,108,167
2022,80,112,167
2023,80,112,169
2024,80,112,167


## 3. Crash target audit
`MAX_SEV`/`MAX_SEVNAME` are used only to construct and explain the target. `NUM_INJ`, `MAXSEV_IM`, `NO_INJ_IM` and related injury/outcome fields must not enter the predictors. The following code measures the annual class distribution without altering the raw data.

In [4]:
target_rows = []
for year in CRSS_YEARS:
    path = crss_extracted_dir(year) / 'accident.csv'
    df = pd.read_csv(path, usecols=['CASENUM','MAX_SEV','MAX_SEVNAME','WEIGHT'], low_memory=False)
    labels = df['MAX_SEVNAME'].astype('string').str.strip()
    # Codes/names are inspected before the binary mapping is frozen.
    display(pd.DataFrame({'year': year, 'count': labels.value_counts(dropna=False), 'percent': labels.value_counts(normalize=True, dropna=False).mul(100)}).head(15))
    target_rows.append({'year': year, 'crashes': len(df), 'unique_crashes': df['CASENUM'].nunique(), 'missing_target': labels.isna().sum()})
pd.DataFrame(target_rows)

,year,count,percent
MAX_SEVNAME,,,
No Apparent Injury (O),2020,24204,44.212257
Possible Injury (C),2020,12141,22.177368
Suspected Minor Injury (B),2020,9540,17.426249
Suspected Serious Injury (A),2020,6270,11.453101
Fatal Injury (K),2020,1387,2.533565
Unknown/Not Reported,2020,1017,1.857704
"Injured, Severity Unknown",2020,166,0.303224
No person involved,2020,19,0.034706
Died Prior to Crash*,2020,1,0.001827


,year,count,percent
MAX_SEVNAME,,,
No Apparent Injury (O),2021,24907,45.953875
Possible Injury (C),2021,11922,21.99631
Suspected Minor Injury (B),2021,9321,17.197417
Suspected Serious Injury (A),2021,5787,10.677122
Fatal Injury (K),2021,1185,2.186347
Unknown/Not Reported,2021,929,1.714022
"Injured, Severity Unknown",2021,134,0.247232
No person involved,2021,13,0.023985
Died Prior to Crash*,2021,2,0.00369


,year,count,percent
MAX_SEVNAME,,,
No Apparent Injury (O),2022,25961,48.116023
Possible Injury (C),2022,11118,20.606061
Suspected Minor Injury (B),2022,8703,16.130108
Suspected Serious Injury (A),2022,5786,10.723751
Fatal Injury (K),2022,1284,2.379761
Unknown/Not Reported,2022,975,1.807061
"Injured, Severity Unknown",2022,104,0.192753
No person involved,2022,21,0.038921
Died Prior to Crash*,2022,3,0.00556


,year,count,percent
MAX_SEVNAME,,,
No Apparent Injury (O),2023,23856,47.613915
Possible Injury (C),2023,9947,19.853103
Suspected Minor Injury (B),2023,8635,17.234497
Suspected Serious Injury (A),2023,5495,10.967407
Fatal Injury (K),2023,1130,2.255354
Unknown/Not Reported,2023,925,1.846197
"Injured, Severity Unknown",2023,103,0.205577
No person involved,2023,11,0.021955
Died Prior to Crash*,2023,1,0.001996


,year,count,percent
MAX_SEVNAME,,,
No Apparent Injury (O),2024,24732,47.876418
Possible Injury (C),2024,10139,19.627163
Suspected Minor Injury (B),2024,9198,17.805567
Suspected Serious Injury (A),2024,5489,10.625653
Fatal Injury (K),2024,1096,2.121646
Unknown/Not Reported,2024,857,1.658988
"Injured, Severity Unknown",2024,127,0.245848
No person involved,2024,17,0.032909
Died Prior to Crash*,2024,3,0.005807


,year,crashes,unique_crashes,missing_target
0,2020,54745,54745,0
1,2021,54200,54200,0
2,2022,53955,53955,0
3,2023,50103,50103,0
4,2024,51658,51658,0


## 4. CRSS schema harmonisation audit
Field presence and documented meaning are checked across all five years. Same-named fields are not assumed to have identical codes. Both the analytical manual and the corresponding annual coding manual govern harmonisation.

In [5]:
def header_set(year, table):
    path = crss_extracted_dir(year) / f'{table}.csv'
    return set(pd.read_csv(path, nrows=0).columns)

for table in ['accident','vehicle','person']:
    sets = {year: header_set(year, table) for year in CRSS_YEARS}
    common = set.intersection(*sets.values())
    union = set.union(*sets.values())
    print(table, 'common fields:', len(common), 'all observed fields:', len(union))
    for year in CRSS_YEARS:
        print(' ', year, 'year-specific/missing relative to union:', len(union - sets[year]))

accident common fields: 80 all observed fields: 80
  2020 year-specific/missing relative to union: 0
  2021 year-specific/missing relative to union: 0
  2022 year-specific/missing relative to union: 0
  2023 year-specific/missing relative to union: 0
  2024 year-specific/missing relative to union: 0
vehicle common fields: 163 all observed fields: 173
  2020 year-specific/missing relative to union: 6
  2021 year-specific/missing relative to union: 6
  2022 year-specific/missing relative to union: 6
  2023 year-specific/missing relative to union: 4
  2024 year-specific/missing relative to union: 6
person common fields: 108 all observed fields: 112
  2020 year-specific/missing relative to union: 4
  2021 year-specific/missing relative to union: 4
  2022 year-specific/missing relative to union: 0
  2023 year-specific/missing relative to union: 0
  2024 year-specific/missing relative to union: 0


## 5. Complaint field dictionary and grouping
The official layout documents 51 tab-delimited fields. `CDESCR` is the narrative, `COMPDESC` supplies the target labels and `ODINO` groups repeated component rows. The raw audit found 1,622,118 unique `ODINO` values and 628,186 repeated component rows, confirming that row-wise random splitting would leak complaint information.

In [6]:
COMPLAINT_COLUMNS = [
 'CMPLID','ODINO','MFR_NAME','MAKETXT','MODELTXT','YEARTXT','CRASH','FAILDATE','FIRE','INJURED','DEATHS','COMPDESC','CITY','STATE','VIN','DATEA','LDATE','MILES','OCCURENCES','CDESCR','CMPL_TYPE','POLICE_RPT_YN','PURCH_DT','ORIG_OWNER_YN','ANTI_BRAKES_YN','CRUISE_CONT_YN','NUM_CYLS','DRIVE_TRAIN','FUEL_SYS','FUEL_TYPE','TRANS_TYPE','VEH_SPEED','DOT','TIRE_SIZE','LOC_OF_TIRE','TIRE_FAIL_TYPE','ORIG_EQUIP_YN','MANUF_DT','SEAT_TYPE','RESTRAINT_TYPE','DEALER_NAME','DEALER_TEL','DEALER_CITY','DEALER_STATE','DEALER_ZIP','PROD_TYPE','REPAIRED_YN','MEDICAL_ATTN','VEHICLES_TOWED_YN','STATE_OF_INCIDENT','VEHICLE_OPERATOR'
]
sample = pd.read_csv(complaint_file(), sep='\t', names=COMPLAINT_COLUMNS, dtype='string', nrows=100_000, encoding='latin-1', on_bad_lines='skip')
display(sample[['ODINO','COMPDESC','CDESCR','LDATE','DATEA']].head())
display(sample[['ODINO','COMPDESC','CDESCR','LDATE','DATEA']].isna().mean().mul(100).rename('missing_percent').to_frame())

,ODINO,COMPDESC,CDESCR,LDATE,DATEA
0,958241,ENGINE AND ENGINE COOLING:COOLING SYSTEM:RADIA...,RADIATOR FAILED @ HIGHWAY SPEED OBSTRUCTING DR...,19950103,19950103
1,958130,"FUEL SYSTEM, GASOLINE:DELIVERY","FUEL LEAKED FROM FUEL TANK AREA, EMITTING STRO...",19950103,19950103
2,958132,POWER TRAIN:AUTOMATIC TRANSMISSION,SHIFTED INTO REVERSE VEHICLE JERKED VIOLENTLY....,19950103,19950103
3,958133,"FUEL SYSTEM, GASOLINE:STORAGE:TANK ASSEMBLY",FUEL TANK ; LEAKS BECAUSE OF RUST GAS LEAK BY ...,19950103,19950103
4,958137,SEATS,"DRIVER SIDE SEAT FRAME BROKE IN TWO, CAUSING S...",19950103,19950103


,missing_percent
ODINO,0.0
COMPDESC,0.0
CDESCR,0.0
LDATE,0.0
DATEA,0.0


## 6. Complaint target-design checkpoint
The broad target candidate is the first hierarchy level of `COMPDESC` (text before the first colon). Before freezing the task, the full dataset must establish: label frequencies, labels per `ODINO`, ambiguous/missing dates, duplicate narratives, and whether rare labels require an `OTHER` class or a genuinely multi-label formulation. This decision is made from training-period evidence and domain meaning—not test performance.

## 7. Data Understanding exit criteria
Proceed to preparation only after all items are evidenced:
- every selected predictor has an official definition;
- join keys and table grains are verified;
- target mappings are documented;
- leakage fields are registered;
- annual code differences are resolved or excluded;
- complaint date, duplicate and multi-label behaviour are measured;
- class distributions are known; and
- temporal partitions are frozen.